# Dimensionality Reduction Algorithms: Theory, Math, and Implementation

Dimensionality Reduction is an unsupervised learning technique used to reduce the number of variables (features) in a dataset while retaining as much of the important, underlying information as possible. 

In real-world data science, datasets often have hundreds or thousands of features (high-dimensionality). This causes the **"Curse of Dimensionality,"** making models slow, prone to overfitting, and impossible to visualize. Dimensionality reduction simplifies data, removes noise, and allows us to visualize complex, high-dimensional spaces in 2D or 3D.

---
### Setup & Imports
We will generate multiple types of datasets to test our algorithms:
1. **3D Blobs:** To see how PCA flattens spatial data.
2. **The Digits Dataset:** A 64-dimensional dataset (8x8 pixel images of numbers) to test t-SNE.
3. **Mixed Waveforms:** To test ICA's ability to separate signals.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA, FastICA
from sklearn.manifold import TSNE
from sklearn.datasets import make_blobs, load_digits
from ipywidgets import interact, IntSlider, FloatSlider
import scipy.signal

plt.style.use('seaborn-v0_8-darkgrid')

# 1. Dataset for PCA (3D spatial data)
np.random.seed(42)
X_3d, y_3d = make_blobs(n_samples=300, centers=3, n_features=3, cluster_std=1.5, random_state=42)

# 2. Dataset for t-SNE (64-dimensional Digits data)
# We will use a subset to keep the interactive visualization fast
digits = load_digits()
X_digits = digits.data[:500] 
y_digits = digits.target[:500]

### 1. Principal Component Analysis (PCA)

PCA is a linear dimensionality reduction technique. It rotates and projects the original dataset onto new, orthogonal axes called **Principal Components (PCs)**. 
* **PC1** is the axis that captures the absolute maximum variance (spread) in the data.
* **PC2** is the axis orthogonal (perpendicular) to PC1 that captures the second most variance, and so on.

By keeping only the top few components, you discard noise and highly correlated features while keeping the core structure.

**Mathematical Foundation:**
1. **Standardization:** Center the data by subtracting the mean: $X_{centered} = X - \mu$.
2. **Covariance Matrix:** Calculate how features vary together: $\Sigma = \frac{1}{n-1} X^T X$
3. **Eigendecomposition:** Find the eigenvectors ($v$) and eigenvalues ($\lambda$) of the covariance matrix. 
   $$\Sigma v = \lambda v$$
   * The eigenvectors dictate the *direction* of the new axes.
   * The eigenvalues dictate the *magnitude* (how much variance that axis explains).
4. **Projection:** Sort the eigenvectors by their eigenvalues in descending order. Multiply the original data by the top $k$ eigenvectors to get the compressed data:
   $$X_{pca} = X \cdot W_k$$

**Example Problem:**
* **Neuroscience:** An EEG cap records brain waves from 64 different channels (64 dimensions) simultaneously. PCA compresses this highly correlated data into 2-3 principal components, allowing scientists to visualize specific "brain states" on a simple 2D scatter plot.

The visualization below starts with a 3D dataset. Use the slider to view it from different angles. PCA will automatically figure out the absolute best 2D angle (the "shadow") that preserves the most spread out data.

In [14]:
@interact(elev=IntSlider(min=-90, max=90, step=10, value=20, description='3D Elevation'),
          azim=IntSlider(min=-180, max=180, step=10, value=30, description='3D Azimuth'))
def plot_pca_concept(elev, azim):
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_3d)
    
    fig = plt.figure(figsize=(14, 6))
    
    ax1 = fig.add_subplot(121, projection='3d')
    ax1.scatter(X_3d[:, 0], X_3d[:, 1], X_3d[:, 2], c=y_3d, cmap='viridis', s=40, edgecolor='k')
    ax1.view_init(elev=elev, azim=azim)
    ax1.set_title("Original 3D Data Space", fontsize=14)
    ax1.set_xlabel("Feature 1")
    ax1.set_ylabel("Feature 2")
    ax1.set_zlabel("Feature 3")
    
    ax2 = fig.add_subplot(122)
    ax2.scatter(X_pca[:, 0], X_pca[:, 1], c=y_3d, cmap='viridis', s=50, edgecolor='k')
    
    var_explained = sum(pca.explained_variance_ratio_) * 100
    ax2.set_title(f"2D PCA Projection (Captures {var_explained:.1f}% of Variance)", fontsize=14)
    ax2.set_xlabel("Principal Component 1")
    ax2.set_ylabel("Principal Component 2")
    
    plt.tight_layout()
    plt.show()

interactive(children=(IntSlider(value=20, description='3D Elevation', max=90, min=-90, step=10), IntSlider(val…

### 2. t-SNE & UMAP (Non-Linear Dimensionality Reduction)

t-Distributed Stochastic Neighbor Embedding (t-SNE) is a heavily mathematical, non-linear technique. While PCA is great for global structure, it fails spectacularly if the data is wrapped, twisted, or curved. t-SNE is built specifically to visualize high-dimensional data in 2D or 3D by strictly preserving **local neighborhoods** (keeping similar data points close together).

*(Note: UMAP is a modern alternative to t-SNE. It is mathematically different, drastically faster, and preserves both local AND global structure better, but t-SNE remains an absolute classic).*

**Mathematical Foundation:**
t-SNE treats dimensionality reduction as an optimization problem of probability distributions:
1. **High-Dimensional Space:** It calculates the probability ($p_{ij}$) that point $i$ would pick point $j$ as its neighbor based on a Gaussian distribution.
2. **Low-Dimensional Space:** It creates a similar probability distribution ($q_{ij}$) for the points in the new 2D space, but uses a Student's t-distribution (which has "heavy tails") to prevent the points from crowding together in the center.
3. **Kullback-Leibler (KL) Divergence:** The algorithm shifts the points around in the 2D space to minimize the difference between the high-dim probabilities and the low-dim probabilities using Gradient Descent:
   $$Cost = KL(P || Q) = \sum_{i} \sum_{j} p_{ij} \log \frac{p_{ij}}{q_{ij}}$$

**Example Problem:**
* **Genomics:** Visualizing single-cell RNA sequencing data. An algorithm analyzes thousands of genes (features) per cell and uses t-SNE/UMAP to cluster them into visual islands on a 2D plot, instantly revealing distinct cell types.

**Hyperparameters:**
* **Perplexity:** Think of this as a knob that balances attention between local and global aspects of your data. It represents the estimated number of close neighbors each point has.

In [ ]:
@interact(perplexity=IntSlider(min=5, max=50, step=5, value=30, description='Perplexity'))
def plot_tsne(perplexity):

    tsne = TSNE(n_components=2, perplexity=perplexity, random_state=42, init='pca', learning_rate='auto')
    X_tsne = tsne.fit_transform(X_digits)
    
    plt.figure(figsize=(10, 8))
    scatter = plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_digits, cmap='tab10', s=40, edgecolor='k', alpha=0.8)  
    plt.legend(handles=scatter.legend_elements()[0], labels=[str(i) for i in range(10)], title="Digits")
    
    plt.title(f"t-SNE Projection of 64D Digits Data (Perplexity={perplexity})", fontsize=15)
    plt.xlabel("t-SNE Dimension 1")
    plt.ylabel("t-SNE Dimension 2")
    plt.show()

interactive(children=(IntSlider(value=30, description='Perplexity', max=50, min=5, step=5), Output()), _dom_cl…

### 3. Independent Component Analysis (ICA)

Independent Component Analysis (ICA) is vastly different from PCA. While PCA seeks axes of maximum *variance*, ICA seeks subcomponents that are statistically **independent**. 

The classic explanation of ICA is the **"Cocktail Party Problem."** Imagine you are in a room with three people talking simultaneously, and you have three microphones placed around the room. Each microphone records a messy, mixed signal of all three voices at different volumes. ICA can take those mixed recordings and mathematically unmix them back into the three isolated, independent voices!

**Mathematical Foundation:**
ICA assumes that the observed data ($X$) is a linear combination of unknown, independent source signals ($S$), mixed together by an unknown mixing matrix ($A$):
$$X = A \cdot S$$

The goal of ICA is to find an "unmixing" matrix ($W$, which is approximately $A^{-1}$) so that we can recover the original sources:
$$S_{recovered} = W \cdot X$$

*How does it find $W$?* 

The Central Limit Theorem states that combining independent signals makes the resulting mixture more Gaussian (normal/bell-curve). Therefore, ICA uses optimization algorithms (like FastICA) to find the matrix $W$ that maximizes the **Non-Gaussianity** of the output components.

**Example Problem:**
* **Signal Processing (EEG):** A patient blinks their eyes during an EEG test. The eye muscle electricity massively spikes, ruining the delicate brainwave data. Because the eye blink and the brainwaves are generated by statistically independent sources, ICA can cleanly isolate the eye-blink artifact into a single component, allowing doctors to simply delete it and perfectly reconstruct the clean brainwaves!

In the visualization below, we mathematically mix a Sine wave, a Square wave, and random Noise. Watch how ICA blindly reconstructs the original shapes from the jumbled mess!

In [ ]:
import scipy.signal
from sklearn.decomposition import FastICA

time = np.linspace(0, 8, 2000)
s1 = np.sin(2 * time)  
s2 = np.sign(np.sin(3 * time)) 
s3 = scipy.signal.sawtooth(2 * np.pi * time)  

S = np.c_[s1, s2, s3]
S += 0.2 * np.random.normal(size=S.shape)  
S /= S.std(axis=0)

@interact(mix_strength=FloatSlider(min=0.0, max=1.0, step=0.1, value=0.8, description='Mix Strength'))
def plot_ica_perfected(mix_strength):

    A_identity = np.array([[1.0, 0.0, 0.0], 
                           [0.0, 1.0, 0.0], 
                           [0.0, 0.0, 1.0]])
    
    A_complex = np.array([[1.0, 0.9, 0.4], 
                          [0.5, 1.0, 0.8], 
                          [0.8, 0.3, 1.0]])
    
    A = (1 - mix_strength) * A_identity + mix_strength * A_complex
    

    X = np.dot(S, A.T)
    
    ica = FastICA(n_components=3, random_state=42, max_iter=1000)
    S_recovered = ica.fit_transform(X)  
    S_recovered /= S_recovered.std(axis=0)
    

    S_recovered_matched = np.zeros_like(S_recovered)
    for i in range(S_recovered.shape[1]):
        correlations = [np.abs(np.corrcoef(S[:, j], S_recovered[:, i])[0, 1]) for j in range(S.shape[1])]
        best_match_idx = np.argmax(correlations)
        
        actual_corr = np.corrcoef(S[:, best_match_idx], S_recovered[:, i])[0, 1]
        sign = np.sign(actual_corr)

        S_recovered_matched[:, best_match_idx] = S_recovered[:, i] * sign

    X_plot = X / X.std(axis=0) 
    
    fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
    
    models = [S, X_plot, S_recovered_matched]
    titles = ['1. True Independent Sources (Unknown to Model)', 
              '2. Mixed Observations (What the 3 microphones recorded)', 
              '3. ICA Recovered Signals (Unmixed & Reordered!)']
    
    offsets = [4, 0, -4]
    colors = ['#ff7f0e', '#2ca02c', '#d62728']
    
    for i, (model, title) in enumerate(zip(models, titles)):
        axes[i].set_title(title, fontsize=12, fontweight='bold')
        for j, (sig, c, offset) in enumerate(zip(model.T, colors, offsets)):
            axes[i].plot(time, sig + offset, color=c, alpha=0.9, linewidth=1.5)
        
        axes[i].set_yticks([]) 
            
    plt.tight_layout()
    plt.show()

interactive(children=(FloatSlider(value=0.8, description='Mix Strength', max=1.0), Output()), _dom_classes=('w…